In [2]:
%pip install "tabfm[pytorch]==1.0.1" scikit-learn

  Using cached tabfm-1.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached jaxtyping-0.2.38-py3-none-any.whl.metadata (6.6 kB)
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.6.0
    Uninstalling typeguard-4.6.0:
      Successfully uninstalled typeguard-4.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [tabfm]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inflect 7.6.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.


In [3]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [16]:
from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0

model = tabfm_v1_0_0.load(model_type="classification")

clf = TabFMClassifier(
    model=model,
    max_num_rows=10_000,
    n_estimators=8,
    random_state=0,
)

print("TabFM loaded")

TabFM loaded


In [6]:
from pathlib import Path

project = Path("/content/IFT3700-GRP10-main")

print("Project folder exists:", project.exists())
print("Raw data folder exists:", (project / "data/raw").exists())
print(
    "XGBoost model exists:",
    (project / "milestone2_outputs/models/xgb_tuned_full.joblib").exists()
)

Project folder exists: True
Raw data folder exists: True
XGBoost model exists: True


In [7]:
import os
import sys
from pathlib import Path

project = Path("/content/IFT3700-GRP10-main")
os.chdir(project)
sys.path.insert(0, str(project))

from milestone2.features import build_dataset, TRAIN_SEASONS, TEST_SEASON

data_dir = project / "data" / "raw"

train_df = build_dataset(
    data_dir,
    seasons=TRAIN_SEASONS,
    game_types=(2,),
)

regular_test_df = build_dataset(
    data_dir,
    seasons=[TEST_SEASON],
    game_types=(2,),
)

playoff_test_df = build_dataset(
    data_dir,
    seasons=[TEST_SEASON],
    game_types=(3,),
    strict=False,
)

print("Training rows:", len(train_df))
print("Regular-season test rows:", len(regular_test_df))
print("Playoffs test rows:", len(playoff_test_df))
print("Training columns:", train_df.columns.tolist())

Training rows: 520564
Regular-season test rows: 79485
Playoffs test rows: 4949
Training columns: ['season', 'game_type', 'game_id', 'event_id', 'team', 'game_seconds', 'period', 'x', 'y', 'shot_distance', 'shot_angle', 'shot_type', 'last_event_type', 'last_event_x', 'last_event_y', 'time_since_last_event', 'distance_from_last_event', 'rebound', 'change_in_shot_angle', 'speed', 'empty_net', 'shooter_skaters', 'opponent_skaters', 'skater_diff', 'is_home', 'last_event_same_team', 'score_diff', 'is_goal']


In [10]:
import numpy as np
import pandas as pd
import time

categorical_features = {"shot_type", "last_event_type"}

def clean_features(frame):
    X = frame[FEATURE_COLUMNS].copy()

    for column in FEATURE_COLUMNS:
        if column in categorical_features:
            X[column] = X[column].astype(object)
            X.loc[pd.isna(X[column]), column] = np.nan
        else:
            X[column] = pd.to_numeric(X[column], errors="coerce")

    return X

X_train = clean_features(train_df)
y_train = train_df["is_goal"].astype(int)

xgb_model = make_pipeline(FEATURE_COLUMNS, best_params)

start = time.perf_counter()
xgb_model.fit(X_train, y_train)
xgb_fit_seconds = time.perf_counter() - start

print("XGBoost fit completed.")
print("Training rows:", len(X_train))
print("Fit time (seconds):", round(xgb_fit_seconds, 2))
print("Parameters:", best_params)

XGBoost fit completed.
Training rows: 520564
Fit time (seconds): 16.93
Parameters: {'subsample': 0.9, 'n_estimators': 250, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.07, 'colsample_bytree': 0.9}


In [15]:
from tabfm import TabFMClassifier
import time

clf = TabFMClassifier(
    model=model,
    max_num_rows=1_000,
    n_estimators=4,
    random_state=0,
)

start = time.perf_counter()
clf.fit(X_train, y_train)
tabfm_fit_seconds = time.perf_counter() - start

print("TabFM fit completed.")
print("Fit/context setup time (seconds):", round(tabfm_fit_seconds, 2))

start = time.perf_counter()
probe_probs = clf.predict_proba(X_regular.iloc[:32])
probe_seconds = time.perf_counter() - start

print("Probe prediction shape:", probe_probs.shape)
print("Probe prediction time (seconds):", round(probe_seconds, 2))

TabFM fit completed.
Fit/context setup time (seconds): 9.77
Probe prediction shape: (32, 2)
Probe prediction time (seconds): 651.65


In [7]:
import os
import sys
from pathlib import Path

project = Path("/content/IFT3700-GRP10-main")
os.chdir(project)
sys.path.insert(0, str(project))

import numpy as np
import pandas as pd
import time
from milestone2.features import FEATURE_COLUMNS
from tabfm import TabFMClassifier

categorical_features = {"shot_type", "last_event_type"}

def clean_features(frame):
    X = frame[FEATURE_COLUMNS].copy()

    for column in FEATURE_COLUMNS:
        if column in categorical_features:
            X[column] = X[column].astype(object)
            X.loc[pd.isna(X[column]), column] = np.nan
        else:
            X[column] = pd.to_numeric(X[column], errors="coerce")

    return X

print("Project folder exists:", project.exists())
print("Training data is in memory:", "train_df" in globals())
print("Regular test data is in memory:", "regular_test_df" in globals())

Project folder exists: True
Training data is in memory: False
Regular test data is in memory: False


In [8]:
from milestone2.features import build_dataset, TRAIN_SEASONS, TEST_SEASON

data_dir = project / "data" / "raw"
output_dir = project / "milestone2_outputs" / "task8"
output_dir.mkdir(parents=True, exist_ok=True)

train_df = build_dataset(
    data_dir,
    seasons=TRAIN_SEASONS,
    game_types=(2,),
)
regular_test_df = build_dataset(
    data_dir,
    seasons=[TEST_SEASON],
    game_types=(2,),
)
playoff_test_df = build_dataset(
    data_dir,
    seasons=[TEST_SEASON],
    game_types=(3,),
    strict=False,
)

train_df.to_csv(output_dir / "train.csv.gz", index=False, compression="gzip")
regular_test_df.to_csv(
    output_dir / "regular_test.csv.gz", index=False, compression="gzip"
)
playoff_test_df.to_csv(
    output_dir / "playoff_test.csv.gz", index=False, compression="gzip"
)

print("Training rows:", len(train_df))
print("Regular-season test rows:", len(regular_test_df))
print("Playoffs test rows:", len(playoff_test_df))
print("Datasets saved to:", output_dir)

Training rows: 520564
Regular-season test rows: 79485
Playoffs test rows: 4949
Datasets saved to: /content/IFT3700-GRP10-main/milestone2_outputs/task8


In [11]:
from tabfm import TabFMClassifier
import time

clf = TabFMClassifier(
    model=model,
    max_num_rows=1_000,
    n_estimators=2,
    random_state=0,
)

start = time.perf_counter()
clf.fit(X_train, y_train)
tabfm_fit_seconds = time.perf_counter() - start

print("TabFM fit/context time (seconds):", round(tabfm_fit_seconds, 2))

TabFM fit/context time (seconds): 9.15


In [12]:
import time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def predict_in_chunks(model, X, chunk_size=500, label="Prediction"):
    chunks = []

    for start_index in tqdm(range(0, len(X), chunk_size), desc=label):
        batch = X.iloc[start_index:start_index + chunk_size]
        chunks.append(model.predict_proba(batch)[:, 1])
        del batch
        torch.cuda.empty_cache()

    return np.concatenate(chunks)

start = time.perf_counter()
tabfm_regular_probs = predict_in_chunks(
    clf, X_regular, chunk_size=500, label="Regular season"
)
tabfm_regular_predict_seconds = time.perf_counter() - start

start = time.perf_counter()
tabfm_playoffs_probs = predict_in_chunks(
    clf, X_playoffs, chunk_size=500, label="Playoffs"
)
tabfm_playoffs_predict_seconds = time.perf_counter() - start

output_dir = project / "milestone2_outputs" / "task8"

pd.DataFrame({
    "y_true": y_regular,
    "tabfm_probability": tabfm_regular_probs,
}).to_csv(output_dir / "regular_tabfm_predictions.csv", index=False)

pd.DataFrame({
    "y_true": y_playoffs,
    "tabfm_probability": tabfm_playoffs_probs,
}).to_csv(output_dir / "playoffs_tabfm_predictions.csv", index=False)

print("Regular-season rows:", len(tabfm_regular_probs))
print("Playoffs rows:", len(tabfm_playoffs_probs))
print("Regular-season prediction seconds:", round(tabfm_regular_predict_seconds, 2))
print("Playoffs prediction seconds:", round(tabfm_playoffs_predict_seconds, 2))

Regular season:   0%|          | 0/159 [00:00<?, ?it/s]

Playoffs:   0%|          | 0/10 [00:00<?, ?it/s]

Regular-season rows: 79485
Playoffs rows: 4949
Regular-season prediction seconds: 937.53
Playoffs prediction seconds: 59.41


In [13]:
import json
import time
import pandas as pd

from milestone2.features import FEATURE_COLUMNS
from milestone2.models import make_pipeline

results_path = project / "milestone2_outputs" / "task5_results.json"
results = json.loads(results_path.read_text(encoding="utf-8"))

best_params = {
    key.removeprefix("model__"): value
    for key, value in results["best_parameters"].items()
}

X_train = clean_features(train_df)
y_train = train_df["is_goal"].astype(int)

X_regular = clean_features(regular_test_df)
X_playoffs = clean_features(playoff_test_df)

# Rebuild the tuned XGBoost pipeline in this Colab environment.
xgb_model = make_pipeline(FEATURE_COLUMNS, best_params)

start = time.perf_counter()
xgb_model.fit(X_train, y_train)
xgb_fit_seconds = time.perf_counter() - start

start = time.perf_counter()
xgb_regular_probs = xgb_model.predict_proba(X_regular)[:, 1]
xgb_regular_predict_seconds = time.perf_counter() - start

start = time.perf_counter()
xgb_playoffs_probs = xgb_model.predict_proba(X_playoffs)[:, 1]
xgb_playoffs_predict_seconds = time.perf_counter() - start

output_dir = project / "milestone2_outputs" / "task8"

pd.DataFrame({
    "y_true": y_regular,
    "xgb_probability": xgb_regular_probs,
}).to_csv(output_dir / "regular_xgb_predictions.csv", index=False)

pd.DataFrame({
    "y_true": y_playoffs,
    "xgb_probability": xgb_playoffs_probs,
}).to_csv(output_dir / "playoffs_xgb_predictions.csv", index=False)

timings = {
    "training_rows": len(X_train),
    "regular_test_rows": len(X_regular),
    "playoffs_test_rows": len(X_playoffs),
    "xgboost_fit_seconds": xgb_fit_seconds,
    "xgboost_regular_prediction_seconds": xgb_regular_predict_seconds,
    "xgboost_playoffs_prediction_seconds": xgb_playoffs_predict_seconds,
    "tabfm_fit_context_seconds": tabfm_fit_seconds,
    "tabfm_regular_prediction_seconds": tabfm_regular_predict_seconds,
    "tabfm_playoffs_prediction_seconds": tabfm_playoffs_predict_seconds,
    "tabfm_max_num_rows": 1000,
    "tabfm_n_estimators": 2,
    "tabfm_device": "cuda:0",
}

(output_dir / "timings.json").write_text(json.dumps(timings, indent=2))
print(json.dumps(timings, indent=2))

{
  "training_rows": 520564,
  "regular_test_rows": 79485,
  "playoffs_test_rows": 4949,
  "xgboost_fit_seconds": 17.68894554200051,
  "xgboost_regular_prediction_seconds": 0.421600107998529,
  "xgboost_playoffs_prediction_seconds": 0.03583378500115941,
  "tabfm_fit_context_seconds": 9.151845076999962,
  "tabfm_regular_prediction_seconds": 937.5273918359999,
  "tabfm_playoffs_prediction_seconds": 59.40787103900038,
  "tabfm_max_num_rows": 1000,
  "tabfm_n_estimators": 2,
  "tabfm_device": "cuda:0"
}


In [14]:
from pathlib import Path

output_dir = project / "milestone2_outputs" / "task8"

for name in [
    "regular_tabfm_predictions.csv",
    "playoffs_tabfm_predictions.csv",
    "regular_xgb_predictions.csv",
    "playoffs_xgb_predictions.csv",
    "timings.json",
]:
    path = output_dir / name
    print(name, path.exists())

regular_tabfm_predictions.csv True
playoffs_tabfm_predictions.csv True
regular_xgb_predictions.csv True
playoffs_xgb_predictions.csv True
timings.json True


In [15]:
from pathlib import Path

project = Path("/content/IFT3700-GRP10-main")

for folder_name in ["milestone2_outputs", "milestone2"]:
    folder = project / folder_name
    print(f"\n--- {folder_name} ---")
    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".py", ".ipynb"}:
                print(path.relative_to(project))


--- milestone2_outputs ---
milestone2_outputs/figures/task5_cumulative_goals.png
milestone2_outputs/figures/task5_feature_selection.png
milestone2_outputs/figures/task5_goal_rate_percentile.png
milestone2_outputs/figures/task5_hyperparameter_tuning.png
milestone2_outputs/figures/task5_reliability.png
milestone2_outputs/figures/task5_roc_auc.png
milestone2_outputs/figures/task6_cumulative_goals.png
milestone2_outputs/figures/task6_goal_rate_percentile.png
milestone2_outputs/figures/task6_reliability.png
milestone2_outputs/figures/task6_roc_auc.png
milestone2_outputs/figures/task7_playoffs_cumulative_goals.png
milestone2_outputs/figures/task7_playoffs_goal_rate_percentile.png
milestone2_outputs/figures/task7_playoffs_reliability.png
milestone2_outputs/figures/task7_playoffs_roc_auc.png
milestone2_outputs/figures/task7_regular_cumulative_goals.png
milestone2_outputs/figures/task7_regular_goal_rate_percentile.png
milestone2_outputs/figures/task7_regular_reliability.png
milestone2_outputs/

In [17]:
plot_file = project / "milestone2" / "task6_experiments.py"
print(plot_file.read_text(encoding="utf-8"))

"""Catalogue des expériences de la tâche 6.

Chaque tentative est une fonction décorée par ``@experiment``. La fonction
reçoit le DataFrame d'entraînement (2016/17-2021/22) et retourne le modèle
entraîné ainsi qu'un dictionnaire de paramètres à enregistrer dans W&B. Le
modèle doit accepter ``frame[features]`` dans ``predict_proba``. Le plus simple
est un Pipeline scikit-learn qui contient sa propre préparation des données :
la tâche 7 pourra alors l'utiliser directement sur les données 2023/24.

Pour ajouter une tentative ::

    @experiment(
        "xgb_new_features",
        features=FEATURE_COLUMNS + ["empty_net"],
        description="XGBoost, hyperparamètres de la tâche 5, nouvelles caractéristiques",
        tags=("xgboost", "features-v2"),
    )
    def xgb_new_features(train, context):
        model = make_pipeline(context.features, task5_best_params(context.output_dir))
        model.fit(train[context.features], train["is_goal"])
        return model, {"hyperparameters": task

In [18]:
evaluation_script = project / "milestone2" / "evaluate_test.py"
print(evaluation_script.read_text(encoding="utf-8"))

"""Tâche 7 : évaluer les 5 modèles retenus sur l'ensemble de test 2023/24.

Ce que fait le script : 

1. télécharge les 5 modèles depuis le registre W&B (3 régressions logistiques
   de la tâche 3, le meilleur XGBoost de la tâche 5, le modèle final de la tâche 6);
2. construit les données de test 2023/24 (saison régulière et séries) avec le
   code qui a servi à entraîner chaque modèle (tâche 2 pour les régressions
   logistiques, tâche 4/6 pour les autres), et garde les mêmes tirs pour tous;
3. calcule les métriques (avec intervalles de confiance par bootstrap sur les
   matchs), les mêmes sur la validation 2022/23 pour comparer, et les 8 figures;
4. enregistre tout dans "milestone2_outputs/task7/" et dans une expérience
   W&B "task7_test_evaluation".
"""

from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import log

In [ ]:
import pandas as pd

task8_dir = project / "milestone2_outputs" / "task8"
task7_predictions = project / "milestone2_outputs" / "task7" / "predictions"

for split in ["regular", "playoffs"]:
    print(f"\n--- {split} ---")

    task8_file = task8_dir / f"{split}_tabfm_predictions.csv"
    task8_df = pd.read_csv(task8_file)
    print("TabFM:", task8_df.shape, task8_df.columns.tolist())

    split_dir = task7_predictions / split
    for path in sorted(split_dir.glob("*.csv*")):
        old_df = pd.read_csv(path)
        print(path.name, old_df.shape, old_df.columns.tolist())


--- regular ---
TabFM: (79485, 2) ['y_true', 'tabfm_probability']
hgb_v2.csv.gz (79483, 4) ['game_id', 'event_id', 'is_goal', 'probability']
logreg_angle.csv.gz (79483, 4) ['game_id', 'event_id', 'is_goal', 'probability']
logreg_distance.csv.gz (79483, 4) ['game_id', 'event_id', 'is_goal', 'probability']
logreg_distance_angle.csv.gz (79483, 4) ['game_id', 'event_id', 'is_goal', 'probability']
xgb_tuned_full.csv.gz (79483, 4) ['game_id', 'event_id', 'is_goal', 'probability']

--- playoffs ---
TabFM: (4949, 2) ['y_true', 'tabfm_probability']
hgb_v2.csv.gz (4949, 4) ['game_id', 'event_id', 'is_goal', 'probability']
logreg_angle.csv.gz (4949, 4) ['game_id', 'event_id', 'is_goal', 'probability']
logreg_distance.csv.gz (4949, 4) ['game_id', 'event_id', 'is_goal', 'probability']
logreg_distance_angle.csv.gz (4949, 4) ['game_id', 'event_id', 'is_goal', 'probability']
xgb_tuned_full.csv.gz (4949, 4) ['game_id', 'event_id', 'is_goal', 'probability']


In [21]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(project))

from milestone2.models import make_evaluation_figures

task8_dir = project / "milestone2_outputs" / "task8"
task7_predictions = project / "milestone2_outputs" / "task7" / "predictions"
figures_dir = task8_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

test_frames = {
    "regular": regular_test_df,
    "playoffs": playoff_test_df,
}

for split, test_frame in test_frames.items():
    split_dir = task7_predictions / split
    old_files = sorted(split_dir.glob("*.csv*"))

    if not old_files:
        raise FileNotFoundError(f"No Task 7 prediction files found for {split}")

    base = pd.read_csv(old_files[0]).set_index(["game_id", "event_id"])
    labels = base["is_goal"].astype(int)
    model_probabilities = {}

    for path in old_files:
        old_df = pd.read_csv(path).set_index(["game_id", "event_id"])
        old_df = old_df.reindex(base.index)

        if old_df["probability"].isna().any():
            raise ValueError(f"Missing aligned predictions in {path.name}")
        if not old_df["is_goal"].astype(int).equals(labels):
            raise ValueError(f"Labels do not match in {path.name}")

        model_name = path.name.replace(".csv.gz", "").replace(".csv", "")
        model_probabilities[model_name] = old_df["probability"].to_numpy()

    tabfm = pd.read_csv(task8_dir / f"{split}_tabfm_predictions.csv")
    tabfm_frame = test_frame[["game_id", "event_id", "is_goal"]].copy()

    if len(tabfm_frame) != len(tabfm):
        raise ValueError(f"{split}: TabFM rows do not match the test dataframe")
    if not np.array_equal(tabfm_frame["is_goal"].to_numpy(), tabfm["y_true"].to_numpy()):
        raise ValueError(f"{split}: TabFM labels do not match the test dataframe")

    tabfm_frame["probability"] = tabfm["tabfm_probability"].to_numpy()
    tabfm_frame = tabfm_frame.set_index(["game_id", "event_id"]).reindex(base.index)

    if tabfm_frame["probability"].isna().any():
        raise ValueError(f"{split}: Could not align all TabFM predictions")
    if not tabfm_frame["is_goal"].astype(int).equals(labels):
        raise ValueError(f"{split}: Aligned TabFM labels do not match")

    model_probabilities["tabfm"] = tabfm_frame["probability"].to_numpy()

    figures = make_evaluation_figures(
        labels,
        model_probabilities,
        figures_dir,
        prefix=f"task8_{split}",
    )

    print(f"{split}: created figures")
    print(figures)

regular: created figures
[PosixPath('/content/IFT3700-GRP10-main/milestone2_outputs/task8/figures/task8_regular_roc_auc.png'), PosixPath('/content/IFT3700-GRP10-main/milestone2_outputs/task8/figures/task8_regular_goal_rate_percentile.png'), PosixPath('/content/IFT3700-GRP10-main/milestone2_outputs/task8/figures/task8_regular_cumulative_goals.png'), PosixPath('/content/IFT3700-GRP10-main/milestone2_outputs/task8/figures/task8_regular_reliability.png')]
playoffs: created figures
[PosixPath('/content/IFT3700-GRP10-main/milestone2_outputs/task8/figures/task8_playoffs_roc_auc.png'), PosixPath('/content/IFT3700-GRP10-main/milestone2_outputs/task8/figures/task8_playoffs_goal_rate_percentile.png'), PosixPath('/content/IFT3700-GRP10-main/milestone2_outputs/task8/figures/task8_playoffs_cumulative_goals.png'), PosixPath('/content/IFT3700-GRP10-main/milestone2_outputs/task8/figures/task8_playoffs_reliability.png')]


In [22]:
from milestone2.evaluation import probability_metrics
import pandas as pd

task8_dir = project / "milestone2_outputs" / "task8"
task7_predictions = project / "milestone2_outputs" / "task7" / "predictions"

test_frames = {
    "regular": regular_test_df,
    "playoffs": playoff_test_df,
}

rows = []

for split, test_frame in test_frames.items():
    split_dir = task7_predictions / split
    old_files = sorted(split_dir.glob("*.csv*"))

    base = pd.read_csv(old_files[0]).set_index(["game_id", "event_id"])
    labels = base["is_goal"].astype(int)

    for path in old_files:
        old_df = pd.read_csv(path).set_index(["game_id", "event_id"])
        old_df = old_df.reindex(base.index)

        model_name = path.name.replace(".csv.gz", "").replace(".csv", "")
        metrics = probability_metrics(labels, old_df["probability"].to_numpy())
        rows.append({"split": split, "model": model_name, **metrics})

    tabfm = pd.read_csv(task8_dir / f"{split}_tabfm_predictions.csv")
    tabfm_frame = test_frame[["game_id", "event_id", "is_goal"]].copy()
    tabfm_frame["probability"] = tabfm["tabfm_probability"].to_numpy()
    tabfm_frame = tabfm_frame.set_index(["game_id", "event_id"]).reindex(base.index)

    metrics = probability_metrics(labels, tabfm_frame["probability"].to_numpy())
    rows.append({"split": split, "model": "tabfm", **metrics})

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(task8_dir / "metrics.csv", index=False)

print(metrics_df.to_string(index=False))
print(f"\nSaved metrics to: {task8_dir / 'metrics.csv'}")

   split                 model      auc  log_loss    brier      ece  xg_goal_ratio  accuracy  shots  goals
 regular                hgb_v2 0.763389  0.281752 0.081629 0.014178       1.024621  0.902847  79483   8085
 regular          logreg_angle 0.561038  0.327442 0.091095 0.012725       0.903430  0.898280  79483   8085
 regular       logreg_distance 0.690537  0.312414 0.087805 0.016685       0.949103  0.898280  79483   8085
 regular logreg_distance_angle 0.715921  0.307576 0.086943 0.021416       0.909401  0.898280  79483   8085
 regular        xgb_tuned_full 0.748001  0.292550 0.085059 0.015647       1.012484  0.898444  79483   8085
 regular                 tabfm 0.707750  0.308623 0.087726 0.022281       0.803684  0.898280  79483   8085
playoffs                hgb_v2 0.757658  0.282206 0.080899 0.017657       1.020720  0.905435   4949    504
playoffs          logreg_angle 0.577228  0.326813 0.091057 0.020574       0.898787  0.898161   4949    504
playoffs       logreg_distance 0.6651

In [1]:
import json
import wandb
from milestone2.wandb_tracking import _init_run

timings = json.loads((task8_dir / "timings.json").read_text(encoding="utf-8"))
figure_files = sorted((task8_dir / "figures").glob("task8_*.png"))

run = _init_run(
    "task8_tabfm_comparison",
    "ift3700-milestone-2",
    None,
    "online",
    job_type="evaluation",
    tags=["task-8", "tabfm", "xgboost-comparison"],
    config={
        "test_season": "2023-24",
        "tabfm_max_num_rows": 1000,
        "tabfm_n_estimators": 2,
        "tabfm_device": "cuda:0",
        "prediction_chunk_size": 500,
    },
)

run.log({"task8_metrics": wandb.Table(dataframe=metrics_df)})

for path in figure_files:
    run.log({f"task8/{path.stem}": wandb.Image(str(path))})

run.summary.update({f"timing/{key}": value for key, value in timings.items()})
print("W&B run:", run.url)
run.finish()

ModuleNotFoundError: No module named 'milestone2'

In [2]:
import sys
import torch
from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch

print("Python:", sys.executable)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("TabFM import: OK")

Python: c:\Users\sunjunran\Desktop\IFT3700-GRP10-main\.venv\Scripts\python.exe
CUDA: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
TabFM import: OK


In [3]:
import os
import sys
from pathlib import Path

project = Path(r"C:\Users\sunjunran\Desktop\IFT3700-GRP10-main")
os.chdir(project)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

data_dir = project / "data" / "raw"
results_file = project / "milestone2_outputs" / "task5_results.json"

print("Project exists:", project.exists())
print("Raw data exists:", data_dir.exists())
print("Raw data seasons:", sorted(p.name for p in data_dir.iterdir() if p.is_dir()) if data_dir.exists() else "NOT FOUND")
print("Task 5 results exist:", results_file.exists())

Project exists: True
Raw data exists: True
Raw data seasons: ['20162017', '20172018', '20182019', '20192020', '20202021', '20212022', '20222023', '20232024']
Task 5 results exist: True


In [5]:
from milestone2.features import build_dataset, TRAIN_SEASONS, TEST_SEASON

output_dir = project / "milestone2_outputs" / "task8"
output_dir.mkdir(parents=True, exist_ok=True)

train_df = build_dataset(
    data_dir,
    seasons=TRAIN_SEASONS,
    game_types=(2,),
)

regular_test_df = build_dataset(
    data_dir,
    seasons=[TEST_SEASON],
    game_types=(2,),
)

playoff_test_df = build_dataset(
    data_dir,
    seasons=[TEST_SEASON],
    game_types=(3,),
    strict=False,
)

train_df.to_csv(output_dir / "train.csv.gz", index=False, compression="gzip")
regular_test_df.to_csv(output_dir / "regular_test.csv.gz", index=False, compression="gzip")
playoff_test_df.to_csv(output_dir / "playoff_test.csv.gz", index=False, compression="gzip")

print("Training rows:", len(train_df))
print("Regular-season test rows:", len(regular_test_df))
print("Playoffs test rows:", len(playoff_test_df))
print("Datasets saved to:", output_dir)

Training rows: 520564
Regular-season test rows: 79485
Playoffs test rows: 4949
Datasets saved to: C:\Users\sunjunran\Desktop\IFT3700-GRP10-main\milestone2_outputs\task8


In [6]:
import json
import time
import joblib
import numpy as np
import pandas as pd

from milestone2.features import FEATURE_COLUMNS
from milestone2.models import make_pipeline

categorical_features = {"shot_type", "last_event_type"}

def clean_features(frame):
    X = frame[FEATURE_COLUMNS].copy()

    for column in FEATURE_COLUMNS:
        if column in categorical_features:
            X[column] = X[column].astype(object)
            X.loc[pd.isna(X[column]), column] = np.nan
        else:
            X[column] = pd.to_numeric(X[column], errors="coerce")

    return X

X_train = clean_features(train_df)
y_train = train_df["is_goal"].astype(int)

X_regular = clean_features(regular_test_df)
y_regular = regular_test_df["is_goal"].astype(int)

X_playoffs = clean_features(playoff_test_df)
y_playoffs = playoff_test_df["is_goal"].astype(int)

results = json.loads(results_file.read_text(encoding="utf-8"))
best_params = {
    key.removeprefix("model_"): value
    for key, value in results["best_parameters"].items()
}

xgb_model = make_pipeline(FEATURE_COLUMNS, best_params)

start = time.perf_counter()
xgb_model.fit(X_train, y_train)
xgb_fit_seconds = time.perf_counter() - start

joblib.dump(xgb_model, output_dir / "xgb_rebuilt.joblib")

print("XGBoost fit completed.")
print("Training rows:", len(X_train))
print("Fit time (seconds):", round(xgb_fit_seconds, 2))
print("Saved model:", output_dir / "xgb_rebuilt.joblib")

c:\Users\sunjunran\Desktop\IFT3700-GRP10-main\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:33:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "_colsample_bytree", "_learning_rate", "_max_depth", "_min_child_weight", "_n_estimators", "_subsample" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost fit completed.
Training rows: 520564
Fit time (seconds): 3.09
Saved model: C:\Users\sunjunran\Desktop\IFT3700-GRP10-main\milestone2_outputs\task8\xgb_rebuilt.joblib


In [2]:
import os
import sys
import joblib
import pandas as pd
from pathlib import Path

project = Path(r"C:\Users\sunjunran\Desktop\IFT3700-GRP10-main")
os.chdir(project)
sys.path.insert(0, str(project))

output_dir = project / "milestone2_outputs" / "task8"

train_df = pd.read_csv(output_dir / "train.csv.gz")
regular_test_df = pd.read_csv(output_dir / "regular_test.csv.gz")
playoff_test_df = pd.read_csv(output_dir / "playoff_test.csv.gz")

model_file = output_dir / "xgb_rebuilt.joblib"
print("Training rows:", len(train_df))
print("Regular test rows:", len(regular_test_df))
print("Playoffs test rows:", len(playoff_test_df))
print("XGBoost model file exists:", model_file.exists())

if model_file.exists():
    xgb_model = joblib.load(model_file)
    print("XGBoost model loaded.")

Training rows: 520564
Regular test rows: 79485
Playoffs test rows: 4949
XGBoost model file exists: True
XGBoost model loaded.


In [3]:
import time
import numpy as np
import pandas as pd
from milestone2.features import FEATURE_COLUMNS

categorical_features = {"shot_type", "last_event_type"}

def clean_features(frame):
    X = frame[FEATURE_COLUMNS].copy()

    for column in FEATURE_COLUMNS:
        if column in categorical_features:
            X[column] = X[column].astype(object)
            X.loc[pd.isna(X[column]), column] = np.nan
        else:
            X[column] = pd.to_numeric(X[column], errors="coerce")

    return X

X_train = clean_features(train_df)
y_train = train_df["is_goal"].astype(int)

X_regular = clean_features(regular_test_df)
y_regular = regular_test_df["is_goal"].astype(int)

X_playoffs = clean_features(playoff_test_df)
y_playoffs = playoff_test_df["is_goal"].astype(int)

start = time.perf_counter()
xgb_regular_probs = xgb_model.predict_proba(X_regular)[:, 1]
xgb_regular_seconds = time.perf_counter() - start

start = time.perf_counter()
xgb_playoffs_probs = xgb_model.predict_proba(X_playoffs)[:, 1]
xgb_playoffs_seconds = time.perf_counter() - start

regular_test_df[["game_id", "event_id", "is_goal"]].assign(
    xgb_probability=xgb_regular_probs
).to_csv(output_dir / "regular_xgb_predictions.csv", index=False)

playoff_test_df[["game_id", "event_id", "is_goal"]].assign(
    xgb_probability=xgb_playoffs_probs
).to_csv(output_dir / "playoffs_xgb_predictions.csv", index=False)

print("XGBoost predictions saved.")
print("Regular-season prediction time:", round(xgb_regular_seconds, 2), "seconds")
print("Playoffs prediction time:", round(xgb_playoffs_seconds, 2), "seconds")

XGBoost predictions saved.
Regular-season prediction time: 0.13 seconds
Playoffs prediction time: 0.01 seconds


In [5]:
import gc
import time
import torch
from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0

gc.collect()
torch.cuda.empty_cache()

model = tabfm_v1_0_0.load(
    model_type="classification",
    device="cuda",
)

clf = TabFMClassifier(
    model=model,
    max_num_rows=1000,
    n_estimators=2,
    random_state=0,
)

start = time.perf_counter()
clf.fit(X_train, y_train)
print("TabFM setup seconds:", round(time.perf_counter() - start, 2))

start = time.perf_counter()
probe_probs = clf.predict_proba(X_regular.iloc[:5])
print("Probe shape:", probe_probs.shape)
print("Probe prediction seconds:", round(time.perf_counter() - start, 2))
print("GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights from local directory
TabFM setup seconds: 15.72
Probe shape: (5, 2)
Probe prediction seconds: 2.0
GPU memory allocated (GB): 3.06


In [6]:
import time
import torch

start = time.perf_counter()
batch_probe = clf.predict_proba(X_regular.iloc[:64])
batch_seconds = time.perf_counter() - start

print("Batch shape:", batch_probe.shape)
print("64-row prediction time (seconds):", round(batch_seconds, 2))
print("GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

Batch shape: (64, 2)
64-row prediction time (seconds): 1.35
GPU memory allocated (GB): 3.06


In [7]:
import math
import time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def predict_and_checkpoint(model, X, frame, destination, chunk_size=64, label="TabFM"):
    key_columns = ["game_id", "event_id"]
    destination.parent.mkdir(parents=True, exist_ok=True)

    done = 0
    if destination.exists():
        saved = pd.read_csv(destination)
        done = len(saved)

        if done > len(frame):
            raise ValueError(f"{destination.name} has more rows than the dataset.")

        saved_keys = saved[key_columns].astype(str).to_numpy()
        current_keys = frame.iloc[:done][key_columns].astype(str).to_numpy()
        if not np.array_equal(saved_keys, current_keys):
            raise ValueError(f"Saved rows in {destination.name} do not match this dataset.")

        if not np.array_equal(
            saved["is_goal"].astype(int).to_numpy(),
            frame.iloc[:done]["is_goal"].astype(int).to_numpy(),
        ):
            raise ValueError(f"Saved labels in {destination.name} do not match.")

    if done == len(frame):
        print(f"{label}: all {done} rows already saved.")
        return 0.0

    start_time = time.perf_counter()

    starts = range(done, len(X), chunk_size)
    for start_idx in tqdm(starts, desc=label):
        end_idx = min(start_idx + chunk_size, len(X))
        probabilities = model.predict_proba(X.iloc[start_idx:end_idx])[:, 1]

        batch = frame.iloc[start_idx:end_idx][key_columns + ["is_goal"]].copy()
        batch["tabfm_probability"] = probabilities
        batch.to_csv(
            destination,
            mode="a",
            header=not destination.exists(),
            index=False,
        )

    return time.perf_counter() - start_time


tabfm_regular_predict_seconds = predict_and_checkpoint(
    clf,
    X_regular,
    regular_test_df,
    output_dir / "regular_tabfm_predictions.csv",
    chunk_size=64,
    label="Regular season",
)

tabfm_playoffs_predict_seconds = predict_and_checkpoint(
    clf,
    X_playoffs,
    playoff_test_df,
    output_dir / "playoffs_tabfm_predictions.csv",
    chunk_size=64,
    label="Playoffs",
)

print("Regular prediction seconds:", round(tabfm_regular_predict_seconds, 2))
print("Playoffs prediction seconds:", round(tabfm_playoffs_predict_seconds, 2))
print("Saved to:", output_dir)

Regular season:   0%|          | 0/1242 [00:00<?, ?it/s]

Playoffs:   0%|          | 0/78 [00:00<?, ?it/s]

Regular prediction seconds: 1779.98
Playoffs prediction seconds: 123.18
Saved to: C:\Users\sunjunran\Desktop\IFT3700-GRP10-main\milestone2_outputs\task8


In [8]:
import numpy as np
import pandas as pd

from milestone2.evaluation import probability_metrics
from milestone2.models import make_evaluation_figures

task7_predictions = project / "milestone2_outputs" / "task7" / "predictions"
figures_dir = output_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

all_metrics = []
key_columns = ["game_id", "event_id"]

def add_aligned_prediction(name, path, probability_column, base_index, labels, probabilities):
    frame = pd.read_csv(path).set_index(key_columns).reindex(base_index)

    if frame["is_goal"].isna().any():
        raise ValueError(f"Could not align all rows for {name}.")
    if not np.array_equal(frame["is_goal"].astype(int).to_numpy(), labels.to_numpy()):
        raise ValueError(f"Goal labels do not match for {name}.")

    probabilities[name] = frame[probability_column].to_numpy()

for split in ["regular", "playoffs"]:
    old_dir = task7_predictions / split
    old_files = sorted(
        path for path in old_dir.glob("*.csv*")
        if "xgb_tuned_full" not in path.name
    )

    if not old_files:
        raise FileNotFoundError(f"No Task 7 predictions found in {old_dir}")

    base = pd.read_csv(old_files[0]).set_index(key_columns)
    labels = base["is_goal"].astype(int)
    model_probabilities = {}

    for path in old_files:
        name = path.name.replace(".csv.gz", "").replace(".csv", "")
        add_aligned_prediction(
            name, path, "probability", base.index, labels, model_probabilities
        )

    add_aligned_prediction(
        "xgb_tuned_full",
        output_dir / f"{split}_xgb_predictions.csv",
        "xgb_probability",
        base.index,
        labels,
        model_probabilities,
    )

    add_aligned_prediction(
        "tabfm",
        output_dir / f"{split}_tabfm_predictions.csv",
        "tabfm_probability",
        base.index,
        labels,
        model_probabilities,
    )

    make_evaluation_figures(
        labels,
        model_probabilities,
        figures_dir,
        prefix=f"task8_{split}",
    )

    for name, probs in model_probabilities.items():
        all_metrics.append({
            "split": split,
            "model": name,
            **probability_metrics(labels, probs),
        })

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(output_dir / "metrics.csv", index=False)

print(metrics_df.to_string(index=False))
print("Figures saved to:", figures_dir)
print("Metrics saved to:", output_dir / "metrics.csv")

   split                 model      auc  log_loss    brier      ece  xg_goal_ratio  accuracy  shots  goals
 regular                hgb_v2 0.763389  0.281752 0.081629 0.014178       1.024621  0.902847  79483   8085
 regular          logreg_angle 0.561038  0.327442 0.091095 0.012725       0.903430  0.898280  79483   8085
 regular       logreg_distance 0.690537  0.312414 0.087805 0.016685       0.949103  0.898280  79483   8085
 regular logreg_distance_angle 0.715921  0.307576 0.086943 0.021416       0.909401  0.898280  79483   8085
 regular        xgb_tuned_full 0.751154  0.291483 0.084650 0.015272       0.991251  0.898179  79483   8085
 regular                 tabfm 0.707973  0.308628 0.087742 0.022421       0.803155  0.898280  79483   8085
playoffs                hgb_v2 0.757658  0.282206 0.080899 0.017657       1.020720  0.905435   4949    504
playoffs          logreg_angle 0.577228  0.326813 0.091057 0.020574       0.898787  0.898161   4949    504
playoffs       logreg_distance 0.6651

In [ ]:
import json
import time
import joblib

from milestone2.features import FEATURE_COLUMNS
from milestone2.models import make_pipeline

results_file = project / "milestone2_outputs" / "task5_results.json"
results = json.loads(results_file.read_text(encoding="utf-8"))

best_params = {
    key.removeprefix("model__"): value
    for key, value in results["best_parameters"].items()
}

print("Task 5 best parameters:", best_params)

xgb_model = make_pipeline(FEATURE_COLUMNS, best_params)

start = time.perf_counter()
xgb_model.fit(X_train, y_train)
xgb_fit_seconds = time.perf_counter() - start

joblib.dump(xgb_model, output_dir / "xgb_rebuilt.joblib")

print("XGBoost refit with Task 5 best parameters.")
print("Fit time (seconds):", round(xgb_fit_seconds, 2))